In [24]:
import pandas as pd
import unicodedata
from pathlib import Path

def normalize(text):
    text = unicodedata.normalize("NFKD", str(text))
    text = text.encode("ASCII", "ignore").decode("utf-8")
    return text.upper().strip()

DATA_DIR = Path("../data").resolve()
input_path = DATA_DIR / "input/ibge/population/tabela4714.csv"

# Pular as 4 primeiras linhas
df = pd.read_csv(
    input_path,
    sep=",",
    skiprows=4,
    encoding="utf-8",
    engine="python"
)


df.columns = ["nivel", "cd_mun", "municipio", "populacao", "unidade"]

# Manter apenas municípios
df = df[df["nivel"] == "MU"]

# Separar cidade e UF
df[["nm_mun", "uf_raw"]] = df["municipio"].str.extract(r"^(.*)\s\((.*)\)$")

# Normalizar nome
df["nm_mun"] = df["nm_mun"].apply(normalize)

# Sigla UF já vem correta
df["sigla_uf"] = df["uf_raw"]

# Converter população
df["populacao"] = (
    df["populacao"]
    .astype(str)
    .str.replace(".", "", regex=False)   # remove separador milhar se existir
    .str.replace(",", ".", regex=False)  # caso venha decimal
    .str.strip()
)

df["populacao"] = pd.to_numeric(df["populacao"], errors="coerce")


# Selecionar final
df_final = df[["cd_mun", "nm_mun", "sigla_uf", "populacao"]]

print(len(df_final))  # deve dar ~5570





5570


In [26]:
#output_csv = DATA_DIR / "output/br_population_municipio_2022.csv"
#df_final.to_csv(output_csv, index=False)
#print("CSV salvo em:", output_csv)


output_parquet = DATA_DIR / "output/br_population_municipio_2022.parquet"

df_final.to_parquet(output_parquet, index=False)

print("Parquet salvo em:", output_parquet)


Parquet salvo em: G:\My Drive\Github\py-2026-map-shaper\data\output\br_population_municipio_2022.parquet
